# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: FAIR² Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the FAIR² dataset using the `mlcroissant` library. All references to dataset structure use explicit `@id` identifiers as per FAIR and Croissant conventions.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f'Dataset name: {metadata.name}\n')
print(f'Description: {metadata.description}\n')
print(f'Version: {metadata.version}\n')
print('Authors (by @id):')
for author in getattr(metadata, 'author', []):
    if isinstance(author, dict) and '@id' in author:
        print('  ', author['@id'])

## 2. Data Overview
Review available record sets (`@id`s), their fields (`@id`s), and columns. We follow Croissant conventions, referencing all structures by their `@id` fields.

In [ ]:
# List all Record Sets and their available fields by @id
print('Record Sets available in the dataset:')
record_sets = []
for rs in dataset.record_sets:
    print(f'- RecordSet @id: {rs.id}')
    record_sets.append(rs.id)
    # List fields in this recordset
    if hasattr(rs, 'fields'):
        print('  Fields:')
        for field in rs.fields:
            print(f'    - Field @id: {field.id} (name: {getattr(field, "name", "")})')
            if hasattr(field, 'columns'):
                print('      Columns:')
                for col in field.columns:
                    print(f'        - Column @id: {col.id} (name: {getattr(col, "name", "")})')
    print()

# Show a preview of each record (first record for each RecordSet)
for rs_id in record_sets:
    print(f'First record example from RecordSet {rs_id}:')
    rs_iter = dataset.records(record_set=rs_id)
    try:
        pprint.pprint(next(rs_iter))
    except StopIteration:
        print('  (No records found)')
    print()

## 3. Data Extraction
Load data from all record sets found above into pandas DataFrames for analysis. Reference the record set and field/column `@id`s from the overview in each access step.

In [ ]:
# Prepare to extract data for each record set using their `@id`s
dataframes = {}
for rs_id in record_sets:
    # Load all records from this record set as a list of dicts
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f'Loaded record set {rs_id} as a DataFrame, shape: {df.shape}')
    else:
        print(f'Record set {rs_id} contains no records.')

# Display the columns of the main tabular record set (choose the one with most columns/rows)
largest_rs = None
max_cols = -1
for rs_id, df in dataframes.items():
    if df.shape[1] > max_cols:
        largest_rs = rs_id
        max_cols = df.shape[1]

if largest_rs:
    print(f'Primary data table: {largest_rs}')
    print('Columns (@id):', dataframes[largest_rs].columns.tolist())
    display(dataframes[largest_rs].head(3))

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records by criteria (e.g., numeric variables), normalizing, and grouping. All fields referenced in operations are by their Croissant `@id`. Adjust as necessary based on what fields exist in the recordset.

In [ ]:
# EDA: Select numeric and group fields by their @id as present in the main recordset

main_df = dataframes[largest_rs]

# Let's heuristically pick commonly present column names ('age', 'interval_between_cancers', etc)
candidates = [c for c in main_df.columns if 'age' in c.lower() or 'interval' in c.lower()]
numeric_field = candidates[0] if candidates else main_df.select_dtypes(include='number').columns[0]

print(f"Using numeric field '@id': {numeric_field}")
print()

# Filtering: e.g., select those cases where age > 60 (or where numeric_field > mean)
try:
    thresh = main_df[numeric_field].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field]) else None
    if thresh is not None:
        filtered_df = main_df[main_df[numeric_field] > thresh].copy()
        print(f"Filtered records with {numeric_field} > {thresh:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by a categorical field (e.g., MSI status or anatomical site)
        group_fields = [c for c in main_df.columns if 'msi' in c.lower() or 'location' in c.lower() or 'sex' in c.lower() or 'anatomical' in c.lower()]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping filtered_df by '{group_field}' (@id):")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean()
            display(grouped)
    else:
        print('No suitable numeric field found for filtering.')
except Exception as e:
    print('Error during EDA step:', e)

## 5. Visualization
Visualize distributions or relationships using the main recordset, referencing each field by its Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the chosen numeric field
plt.figure(figsize=(7,4))
sns.histplot(main_df[numeric_field], kde=True, bins=15)
plt.title(f'Distribution of {numeric_field} (@id)')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group if available
if 'group_field' in locals():
    plt.figure(figsize=(7,4))
    sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
    plt.title(f'{numeric_field} by {group_field} (@id)')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated the step-by-step exploration of the FAIR² dataset with the `mlcroissant` library, referencing all dataset structures by their `@id`. You loaded tabular data, applied example filtering and grouping operations on key clinicopathological variables, and visualized distributions relevant to colorectal cancer survivor analysis.

**Key takeaways:**
- The dataset is well-structured and findable using the Croissant schema and `@id` linking.
- Exploratory steps can fluidly reference variables by their persistent identifiers in Croissant.
- The notebook template is portable for any Croissant-compliant dataset by adjusting the schema URL and field `@id`s.

Explore further analyses or modelling as your use case requires!